In [1]:
import numpy as np
import pandas as pd
import cv2
from matplotlib import pyplot as plt
from numpy.linalg import inv
import os
from os import listdir
from os.path import isfile, isdir, join
from os import walk

from tensorflow import keras
from tensorflow.keras import layers
import tensorflow as tf

import time

# Create transfer function and test

In [2]:
nbi_weights = []

for i in range(6):
    nbi_weights.append(np.load('nor_wli_weight%d.npy'%i))

In [3]:
def sToneCurve(frame):
    look_up_table = np.zeros((256,1), dtype = 'uint8')
    for i in range(256):
        look_up_table[i][0] = 255 * (np.sin(np.pi * (i/255 - 1/2)) + 1) / 2
    return cv2.LUT(frame, look_up_table)

def to_nbi(img, gain = 1.3, is_scurve = True):
    
    img_shape = img.shape
    
    tar_img = img.reshape(-1,3) / 255
    
    tar_img = np.dot(tar_img, nbi_weights[0]) + nbi_weights[1]
    tar_img = np.dot(tar_img, nbi_weights[2]) + nbi_weights[3]
    tar_img = np.dot(tar_img, nbi_weights[4]) + nbi_weights[5]

    tar_img = tar_img * 255 * gain

    tar_img = tar_img.reshape(img_shape)
    
    tar_img[tar_img>255] = 255
    tar_img[tar_img<0] = 0
    
    tar_img = tar_img.astype('uint8')
    
    if is_scurve:    
        tar_img = sToneCurve(tar_img)
    
    return tar_img

# Create LUT table and test 

In [4]:
table_index = np.zeros(256*256*256, dtype = 'int')

for r in range(256):    
    for g in range(256):
        
        b_img = np.zeros((256,3), dtype = 'uint8')
        
        for b in range(256):
            
            b_img[b,:] = np.array([r,g,b], dtype = 'uint8')
            
        to_b_img = to_nbi(b_img)
        
        to_b_img = to_b_img.astype('int')
        to_b_img_keys = to_b_img[:,0] + np.left_shift(to_b_img[:,1], 8) + np.left_shift(to_b_img[:,2], 16)
        
        for b in range(256):
            
            from_v = r + (g<<8) + (b<<16)
            table_index[from_v] = to_b_img_keys[b]
            
print('table_index shape', table_index.shape)

table_index shape (16777216,)


In [5]:

dump_table_file = 'nor_wli_nbi_table.npy'
np.save(dump_table_file, table_index)

# Define table function

In [6]:
table_file = 'nor_wli_nbi_table.npy'
nbi_table = np.load(table_file)

In [7]:
def transfer_by_table(img, table):
    
    # reshape as nx3, and to int
    temp_img = img.reshape(-1, 3).astype(int)

    # transfer to table key
    img_keys = temp_img[:,0] + np.left_shift(temp_img[:,1], 8) + np.left_shift(temp_img[:,2], 16)

    # table mapping
    img_values = table[img_keys]

    # reconstruct to rgb image
    img_r = img_values.astype('uint8')
    img_g = np.right_shift(img_values, 8).astype('uint8')
    img_b = np.right_shift(img_values, 16).astype('uint8')

    img_trans = np.column_stack((img_r, img_g, img_b))

    img_trans = img_trans.reshape(img.shape)
    
    return img_trans

# Batch transfer

In [8]:
def filter_ext(files, tar_ext):
    
    tar_files = []
    
    for file in files:
        
        file_ext = os.path.splitext(file)[-1][1:].lower()
        
        if file_ext == tar_ext:
            tar_files.append(file)
        
    return tar_files
    
def get_all_ext_list(tar_path, tar_ext):
    
    all_tar_files = []
    
    for root, dirs, files in walk(tar_path):
        
        tar_files = filter_ext(files, tar_ext.lower())

        if len(tar_files) == 0:
            continue
            
        tar_files = [join(root, file) for file in tar_files]
            
        all_tar_files = all_tar_files + tar_files
        
    return all_tar_files

In [9]:
def wli_to_nbi(wli, trans_table, is_dual = False):
    
    h,w,c = wli.shape

    image_rgb = cv2.cvtColor(wli, cv2.COLOR_BGR2RGB)
    
    pred_img2 = transfer_by_table(image_rgb, trans_table)

    pred_img2 = cv2.cvtColor(pred_img2, cv2.COLOR_RGB2BGR)
    
    if is_dual:
        tar_img = np.zeros((h, w*2, c))
        tar_img[:h,:w,:] = wli
        tar_img[:h,w:2*w,:] = pred_img2
    else:
        tar_img = np.zeros((h, w, c))
        tar_img[:h,:w,:] = pred_img2
        
    tar_img = tar_img.astype('uint8')
    
    return tar_img

In [128]:
src_path = r'G:\New_datasets_SAVE\Soniya\WLI\train\Ulceration'

tar_path = r'G:\New_datasets_SAVE\Soniya\SAVE\train\Ulceration'

In [129]:
tar_files = get_all_ext_list(src_path, 'jpg')
print('file num: ', len(tar_files))

file num:  347


In [130]:
is_dual = False

table_file = 'nor_wli_nbi_table.npy'
nbi_table = np.load(table_file)

for index, file in enumerate(tar_files):
    
    dump_file = file.replace(src_path, tar_path)
    dump_file_path = os.path.dirname(os.path.abspath(dump_file))
    
    if not os.path.isdir(dump_file_path):
        try:
            os.makedirs(dump_file_path)
        except FileExistsError:
            print("建立路徑異常。")
            
    img = cv2.imread(file)
    nbi = wli_to_nbi(img, nbi_table, is_dual=is_dual)
    
    cv2.imwrite(dump_file, nbi)
    
    print(index, dump_file)

0 G:\New_datasets_SAVE\Soniya\SAVE\train\Ulceration\00021_aug_114.jpg
1 G:\New_datasets_SAVE\Soniya\SAVE\train\Ulceration\00022.jpg
2 G:\New_datasets_SAVE\Soniya\SAVE\train\Ulceration\00023.jpg
3 G:\New_datasets_SAVE\Soniya\SAVE\train\Ulceration\00027_aug_138.jpg
4 G:\New_datasets_SAVE\Soniya\SAVE\train\Ulceration\00027_aug_140.jpg
5 G:\New_datasets_SAVE\Soniya\SAVE\train\Ulceration\00035_aug_28.jpg
6 G:\New_datasets_SAVE\Soniya\SAVE\train\Ulceration\00037.jpg
7 G:\New_datasets_SAVE\Soniya\SAVE\train\Ulceration\00072.jpg
8 G:\New_datasets_SAVE\Soniya\SAVE\train\Ulceration\00159.jpg
9 G:\New_datasets_SAVE\Soniya\SAVE\train\Ulceration\00160.jpg
10 G:\New_datasets_SAVE\Soniya\SAVE\train\Ulceration\00183.jpg
11 G:\New_datasets_SAVE\Soniya\SAVE\train\Ulceration\00194.jpg
12 G:\New_datasets_SAVE\Soniya\SAVE\train\Ulceration\00194_aug_0.jpg
13 G:\New_datasets_SAVE\Soniya\SAVE\train\Ulceration\00194_aug_1.jpg
14 G:\New_datasets_SAVE\Soniya\SAVE\train\Ulceration\00194_aug_2.jpg
15 G:\New_datase

In [19]:
import pyrealsense2 as rs
import numpy as np
import cv2
import time

def order_points(pts):
    """
    Orders 4 points in the order: top-left, top-right, bottom-right, bottom-left.
    """
    rect = np.zeros((4, 2), dtype="float32")
    s = pts.sum(axis=1)
    rect[0] = pts[np.argmin(s)]
    rect[2] = pts[np.argmax(s)]
    
    diff = np.diff(pts, axis=1)
    rect[1] = pts[np.argmin(diff)]
    rect[3] = pts[np.argmax(diff)]
    
    return rect

def get_3d_point(pt, depth_frame, depth_image, intr, depth_scale):
    """
    Averages a small neighborhood around a pixel to reduce noise,
    then deprojects the pixel to a 3D point.
    """
    x, y = int(pt[0]), int(pt[1])
    h, w = depth_image.shape
    xmin = max(x - 2, 0)
    xmax = min(x + 3, w)
    ymin = max(y - 2, 0)
    ymax = min(y + 3, h)
    region = depth_image[ymin:ymax, xmin:xmax]
    valid = region[region > 0]
    if valid.size:
        depth_val = np.median(valid) * depth_scale
    else:
        depth_val = depth_frame.get_distance(x, y)
    return rs.rs2_deproject_pixel_to_point(intr, [x, y], depth_val)

def main():
    # Configure depth and color streams
    pipeline = rs.pipeline()
    config = rs.config()
    config.enable_stream(rs.stream.depth, 840, 480, rs.format.z16, 30)
    config.enable_stream(rs.stream.color, 840, 480, rs.format.bgr8, 30)

    # Start streaming
    try:
        profile = pipeline.start(config)
    except Exception as e:
        print("Could not start pipeline:", e)
        return

    # Get depth scale and camera intrinsics
    depth_sensor = profile.get_device().first_depth_sensor()
    depth_scale = depth_sensor.get_depth_scale()
    align_to = rs.stream.color
    align = rs.align(align_to)
    color_profile = rs.video_stream_profile(profile.get_stream(rs.stream.color))
    intr = color_profile.get_intrinsics()

    try:
        while True:
            # Try to get frames with a timeout
            try:
                frames = pipeline.wait_for_frames(timeout_ms=15000)
            except RuntimeError as e:
                print("Frame timeout error:", e)
                # Optionally, you could continue or break here:
                break

            aligned_frames = align.process(frames)
            depth_frame = aligned_frames.get_depth_frame()
            color_frame = aligned_frames.get_color_frame()
            if not depth_frame or not color_frame:
                continue

            depth_image = np.asanyarray(depth_frame.get_data())
            color_image = np.asanyarray(color_frame.get_data())

            # Process the color image
            gray = cv2.cvtColor(color_image, cv2.COLOR_BGR2GRAY)
            blurred = cv2.GaussianBlur(gray, (5, 5), 0)
            edged = cv2.Canny(blurred, 50, 150)
            contours, _ = cv2.findContours(edged.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            
            if contours:
                c = max(contours, key=cv2.contourArea)
                epsilon = 0.02 * cv2.arcLength(c, True)
                approx = cv2.approxPolyDP(c, epsilon, True)
                
                if len(approx) == 4:
                    pts = approx.reshape(4, 2)
                    rect = order_points(pts)
                    cv2.drawContours(color_image, [np.int0(rect)], -1, (0, 255, 0), 2)
                    
                    pt_tl = get_3d_point(rect[0], depth_frame, depth_image, intr, depth_scale)
                    pt_tr = get_3d_point(rect[1], depth_frame, depth_image, intr, depth_scale)
                    pt_bl = get_3d_point(rect[3], depth_frame, depth_image, intr, depth_scale)
                    
                    width = np.linalg.norm(np.array(pt_tr) - np.array(pt_tl))
                    height = np.linalg.norm(np.array(pt_bl) - np.array(pt_tl))
                    
                    cv2.putText(color_image, f"Width: {width:.2f} m", (10, 30),
                                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)
                    cv2.putText(color_image, f"Height: {height:.2f} m", (10, 70),
                                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)

            cv2.imshow("RealSense Color", color_image)
            cv2.imshow("Edges", edged)
            
            # Exit on 'q' key
            if cv2.waitKey(1) & 0xFF == ord("q"):
                break

    finally:
        # Stop the pipeline
        pipeline.stop()
        # In headless environments, cv2.destroyAllWindows may not be implemented.
        try:
            cv2.destroyAllWindows()
        except Exception as e:
            print("cv2.destroyAllWindows() error:", e)

if __name__ == "__main__":
    main()


Could not start pipeline: No device connected
